# Methane dimer at 36 qubits - SQD on qBraid Lab

Reproducing the 36-qubit experiment in [arXiv:2410.09209](https://arxiv.org/abs/2410.09209),
*Accurate quantum-centric simulations of supramolecular interactions* (Commun. Phys. 2025).

**CAS(16e,16o)/aug-cc-pVQZ | 165,636,900 determinants | 32 + 4 ancilla = 36 qubits**

Written for a **qBraid Pro "Large" session machine (8 vCPU / 25 GB)**. That is
the smallest qBraid tier that can run all of it - section 10 (CASCI) alone needs
~15 GiB, so Free (4 GB) and Standard (8 GB) cannot finish.

---

## How to run this

**Part A** builds a persistent environment. Run it **once**, in whatever kernel
this notebook opened with. It takes a few minutes.

**Then switch kernels** - `Kernel -> Change Kernel... -> Python 3 [methane36]` -
and run **Part B** from the top.

Part A is idempotent (re-running it is a no-op) and Part B refuses to start in
the wrong kernel with a message telling you what to do, so there is no way to
get a confusing failure out of the order you run things in.

Why a venv rather than `%pip install`: on qBraid, packages installed at the
top level do not survive a session, and this pipeline runs for hours. `~` is
persistent storage, so `~/methane_env` is still there tomorrow.

# Part A - one-time setup

Skip to Part B if you already see **Python 3 [methane36]** in the kernel picker
at the top right.

## A1 - Machine and interpreter

Reports the machine and picks an interpreter for the environment. ffsim 0.0.84
requires Python >= 3.11; this looks for a suitable one rather than assuming the
default kernel qualifies.

In [ ]:
import os, platform, shutil, subprocess, sys

VENV   = os.path.expanduser('~/methane_env')
IN_ENV = sys.prefix == VENV

print('kernel python ', sys.version.split()[0], f'({sys.prefix})')
print('platform      ', platform.platform())
if IN_ENV:
    print()
    print('Already running in methane36 - Part A is done, skip to Part B.')

ram_gib  = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
disk_gib = shutil.disk_usage(os.path.expanduser('~')).free / 2**30
print(f'RAM            {ram_gib:.1f} GiB')
print(f'CPUs           {os.cpu_count()}')
print(f'~ free         {disk_gib:.1f} GiB')

assert platform.system() == 'Linux', 'pyscf and ffsim publish no Windows wheels'

# Pick the interpreter that will build the venv: this kernel if it is new
# enough, otherwise the newest python3.x on PATH.
INTERP = sys.executable if sys.version_info >= (3, 11) else None
if INTERP is None:
    for name in ('python3.14', 'python3.13', 'python3.12', 'python3.11'):
        found = shutil.which(name)
        if found:
            INTERP = found
            break
assert INTERP, (
    f'No Python >= 3.11 found (this kernel is {sys.version.split()[0]}), and '
    'ffsim 0.0.84 requires one. Start a newer qBraid image, or install a newer '
    'Python, then re-run this cell.')

version = subprocess.run([INTERP, '-c', 'import sys; print(sys.version.split()[0])'],
                         capture_output=True, text=True).stdout.strip()
print()
print(f'building the environment with {INTERP}  (Python {version})')
print('venv target  ', VENV, '(already exists)' if os.path.isdir(VENV) else '(will be created)')

if ram_gib < 15:
    print()
    print(f'WARNING: {ram_gib:.1f} GiB. Section 10 (CASCI) needs ~15 GiB and will')
    print('         refuse to run. Use a Pro "Large" machine (8 vCPU / 25 GB).')

## A2 - Build the environment

Creates `~/methane_env`, installs the pinned stack, and registers a Jupyter
kernel called **methane36**. Output streams below; the pyscf and ffsim wheels
are large, so give it a few minutes.

Safe to re-run: if the venv already works, it does nothing.

In [ ]:
NEEDED = ['pyscf', 'ffsim', 'qiskit', 'qiskit_addon_sqd', 'ipykernel']
probe  = f'{VENV}/bin/python'
ready  = os.path.isfile(probe) and subprocess.run(
    [probe, '-c', 'import ' + ', '.join(NEEDED)],
    capture_output=True).returncode == 0

if ready:
    print('~/methane_env already has the full stack; nothing to build.')
else:
    print('building; this takes a few minutes...')
    !{INTERP} -m venv {VENV}
    !{VENV}/bin/python -m pip install -q --upgrade pip
    !{VENV}/bin/python -m pip install pyscf==2.14.0 ffsim==0.0.84 'qiskit>=2.0,<3' qiskit-addon-sqd==0.13.1 ipykernel
    !{VENV}/bin/python -m ipykernel install --user --name methane36 --display-name "Python 3 [methane36]"

## A3 - Verify Part A

Checks the venv from the outside before you switch into it, so a bad install is
caught here rather than halfway through section 6.

In [ ]:
check = subprocess.run([f'{VENV}/bin/python', '-c', '''
import ffsim, pyscf, qiskit, qiskit_addon_sqd, sys
print("python", sys.version.split()[0])
print("pyscf", pyscf.__version__)
print("ffsim", ffsim.__version__)
print("qiskit", qiskit.__version__)
print("sqd", qiskit_addon_sqd.__version__)
expected = {"pyscf": "2.14.0", "ffsim": "0.0.84", "qiskit_addon_sqd": "0.13.1"}
actual = {"pyscf": pyscf.__version__, "ffsim": ffsim.__version__,
          "qiskit_addon_sqd": qiskit_addon_sqd.__version__}
drift = {k: (v, actual[k]) for k, v in expected.items() if actual[k] != v}
assert not drift, f"version drift (expected, got): {drift}"
assert qiskit.__version__.startswith("2."), qiskit.__version__
'''], capture_output=True, text=True)

print(check.stdout)
if check.returncode != 0:
    print(check.stderr)
    raise RuntimeError(
        'The environment did not build cleanly. Re-run A2 - the usual cause is '
        'a dropped download. If it persists, delete it and start over:\n'
        f'    !rm -rf {VENV}')

spec = os.path.expanduser('~/.local/share/jupyter/kernels/methane36/kernel.json')
if not os.path.isfile(spec):
    listing = subprocess.run([f'{VENV}/bin/python', '-m', 'jupyter', 'kernelspec', 'list'],
                             capture_output=True, text=True).stdout
    assert 'methane36' in listing, (
        f'kernel not registered at {spec}; re-run A2. jupyter reports:\n{listing}')
print('kernel spec ', spec)

print('environment OK, kernel registered.')
print()
print('=' * 68)
print('  NOW: Kernel -> Change Kernel... -> "Python 3 [methane36]"')
print('  Then run Part B from the top. You never need Part A again.')
print('=' * 68)

---
# Part B - the run

**Switch to the `Python 3 [methane36]` kernel before running anything below.**
B0 checks this and stops with instructions if you have not.

## B0 - Kernel check, resources, pyscf memory

`PYSCF_MAX_MEMORY` matters: pyscf defaults itself to 4000 MB whatever the machine
has, which pushes the aug-cc-pVQZ density fitting out to disk. It has to be set
before pyscf is first imported, and `run.py` subprocesses inherit it.

In [ ]:
import os, platform, shutil, sys

VENV = os.path.expanduser('~/methane_env')
if sys.prefix != VENV:
    raise RuntimeError(
        'Wrong kernel.\n\n'
        f'  running in : {sys.prefix}\n'
        f'  need       : {VENV}\n\n'
        'Kernel -> Change Kernel... -> "Python 3 [methane36]", then re-run '
        'this cell. (If that kernel is not listed, run Part A first.)')

ram_gib = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
print('kernel  ', sys.version.split()[0], '(methane36)')
print(f'RAM      {ram_gib:.1f} GiB')
print(f'CPUs     {os.cpu_count()}')
print(f'~ free   {shutil.disk_usage(os.path.expanduser("~")).free / 2**30:.1f} GiB')

os.environ['PYSCF_MAX_MEMORY'] = str(int(max(2000, ram_gib * 1024 * 0.65)))
print(f"PYSCF_MAX_MEMORY {os.environ['PYSCF_MAX_MEMORY']} MB (pyscf's own default is 4000)")

CAN_CASCI = ram_gib >= 15
if not CAN_CASCI:
    print()
    print(f'NOTE: {ram_gib:.1f} GiB. Sections B1-B8 and B10 will run; section B9')
    print('      (CASCI) needs ~15 GiB and will stop rather than be OOM-killed.')

## B1 - Get the code

Clones into `~/qubit_run`, which is persistent storage. Safe to re-run: it pulls
instead of re-cloning.

In [ ]:
import subprocess

CHECKOUT = os.path.expanduser('~/qubit_run')
ROOT     = os.path.join(CHECKOUT, 'studies/methane-dimer-36q')

if os.path.isdir(os.path.join(CHECKOUT, '.git')):
    subprocess.run(['git', '-C', CHECKOUT, 'pull', '--ff-only'], check=False)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/bhargav2603/qubit_run.git', CHECKOUT], check=True)

assert os.path.isdir(ROOT), f'{ROOT} not found after clone'

os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print('cwd =', os.getcwd())
print(sorted(f for f in os.listdir('.') if f.endswith('.py')))

## B2 - Self-test and cost model

Neither needs the chemistry stack. Run them first: they catch version drift
before any CPU time is spent. Seconds.

In [ ]:
!{sys.executable} run.py selftest --quiet
!{sys.executable} run.py plan

## B3 - END-TO-END VALIDATION (run this before anything expensive)

Runs the **entire chemistry pipeline** - RHF, AVAS, cache round-trip, CCSD, LUCJ,
ffsim sampling, SQD, variance, ablation, binding energy - on a tiny STO-3G active
space. Same molecule, same functions, seconds to run.

The load-bearing check: when the subspace saturates the CAS, **SQD must equal
CASCI to 1e-8 Ha** - solver precision, not chemical accuracy. If the SQD and
CASCI Hamiltonians ever differ, this catches it immediately.

**If this fails, stop. Nothing below will be right.**

In [ ]:
!{sys.executable} run.py validate

## B4 - Geometry

The paper publishes no coordinates and never names the monomer orientation.
D3d is the default; change it here and every downstream cache key changes with it.

In [ ]:
ORIENTATION = 'd3d'
DISTANCE    = 3.638   # the paper's extra point, and its equilibrium

import geometry, paper
atoms = geometry.methane_dimer(DISTANCE, orientation_name=ORIENTATION)
geometry.check_geometry(atoms, expected_distance=DISTANCE)
print(geometry.to_xyz(atoms, f'methane dimer {ORIENTATION} R={DISTANCE}'))

## B5 - Hamiltonian: RHF/aug-cc-pVQZ + AVAS

528 basis functions, so density fitting is on. AVAS on `C[2s,2p], H[1s]` must
land on exactly (16e,16o); `chemistry.py` asserts it rather than tuning to it.

**First run: tens of minutes.** Re-runs hit `data/cache` and return immediately -
and since `~` persists on qBraid, that cache is still there next session.

In [ ]:
from pathlib import Path
import chemistry, spaces

CACHE = Path('data/cache')
spec  = chemistry.SystemSpec(distance=DISTANCE, orientation=ORIENTATION)
mol_data = chemistry.load_or_build(spec, CACHE, verbose=4)

print(f'HF           {mol_data.hf_energy:.10f} Ha')
print(f'active       ({sum(mol_data.nelec)}e,{mol_data.norb}o)')
print(f'determinants {spaces.n_determinants(mol_data.norb, mol_data.nelec):,}')

## B6 - CCSD amplitudes and the LUCJ ansatz

Cheap: `MolecularData.scf` round-trips the *active-space* integrals through an
FCIDUMP, so this is a 16-orbital CCSD, not a 528-orbital one.

In [ ]:
import ansatz, reference

e_ccsd = reference.run_ccsd(mol_data, store_amplitudes=True)
print(f'CCSD (active space) {e_ccsd:.10f} Ha')

layout = ansatz.heavy_hex_layout(mol_data.norb, n_reps=2)
ansatz.validate_layout(layout)      # asserts 32 + 4 = 36
print(layout.to_dict())

operator = ansatz.build_operator(mol_data, layout)
circuit  = ansatz.build_circuit(mol_data, operator)
print(f'circuit: {circuit.num_qubits} qubits, depth {circuit.depth()}')

## B7 - Sample

`FfsimSampler` implements the SamplerV2 interface but simulates inside the (8,8)
sector: 2.47 GiB rather than the 64 GiB a dense 32-qubit statevector would need.

Swap in `sampling.HardwareSampler(backend)` for a real device. Nothing downstream
changes.

In [ ]:
import sampling

SHOTS = paper.TOTAL_SAMPLES      # 200,000
sampler = sampling.NoiselessSampler(mol_data.norb, mol_data.nelec, seed=12345)
sampled = sampler.sample(circuit, SHOTS)

valid    = sampling.valid_configuration_fraction(sampled.bit_array, mol_data.norb, mol_data.nelec)
baseline = sampling.random_validity_probability(mol_data.norb, mol_data.nelec)
print(f'valid {valid:.2%}   random baseline {baseline:.2%}')
print()
print('NOTE: at half filling the random baseline is percent-level, so validity')
print('      fraction is a weak diagnostic here. B10 is the real control.')

## B8 - SQD

Start on the `extrapolation-low` rung (|chi_b| = 9e3). The `converged` rung
reproduces Table II verbatim but needs far more memory - see `run.py plan`.

The addon's default solver runs batches **sequentially**, so peak memory is one
batch rather than ten - but it also means 10 batches x 10 recovery steps = 100
diagonalizations back to back. On 8 vCPUs budget a few hours. Progress prints
per iteration, so you can watch it move.

In [ ]:
import sqd

rung   = spaces.rung('extrapolation-low')
config = sqd.SqdConfig(samples_per_batch=rung.samples_per_batch,
                       n_batches=rung.n_batches,
                       max_iterations=paper.RECOVERY_STEPS,
                       max_dim=rung.max_dim, seed=12345)

result = sqd.run(mol_data, sampled.bit_array, config, source='noiseless')
print()
print(f'SQD  E = {result.energy:.10f} Ha')
print(f'     d = {result.subspace_dimension:,} ({result.subspace_fraction:.2%} of CAS)')

## B9 - CASCI reference (needs the Large machine)

165,636,900 determinants. One CI vector is 961 MiB and Davidson holds a dozen;
`run_casci` sets pyscf's `max_memory` to ~11.9 GB to keep it on the fast path,
so budget ~15 GiB. The guard stops rather than letting the kernel be OOM-killed
hours in.

In [ ]:
import binding

if not CAN_CASCI:
    raise MemoryError(
        f'CASCI(16e,16o) needs ~15 GiB; this machine has {ram_gib:.1f} GiB. '
        'Switch to a qBraid Pro "Large" session machine (8 vCPU / 25 GB) and '
        're-run Part B. The cache in ~/qubit_run survives, so B5 will be instant.')
assert 'result' in globals(), 'run B8 first; this compares against its energy'

e_casci = reference.run_casci(mol_data, verbose=4)
print(f'CASCI {e_casci:.10f} Ha')

binding.variational_check(result.energy, e_casci)   # SQD may not fall below CASCI
agreement = binding.Agreement(result.energy, e_casci)
print(agreement)
print(f'paper target at |chi_b|=20e3: {paper.SQD_VS_CASCI_TARGET_KCAL} kcal/mol')

## B10 - Ablation: the measurement that matters

Uniform random configurations at **matched subspace dimension**. The energy gap
is the quantum layer's contribution, as a number. A control at a different
dimension compares two things at once and settles nothing.

Does not need CASCI, so this one is reachable even if B9 stopped.

In [ ]:
assert 'result' in globals(), 'run B8 first'

control_cfg = sqd.ablation_config(result, config)
uniform     = sampling.UniformSampler(mol_data.norb, mol_data.nelec, seed=12345)
control     = sqd.run(mol_data, uniform.sample(shots=SHOTS).bit_array,
                      control_cfg, source='uniform')

gap = (control.energy - result.energy) / binding.MILLIHARTREE
print(f'SQD      {result.energy:.10f} Ha   d = {result.subspace_dimension:,}')
print(f'uniform  {control.energy:.10f} Ha   d = {control.subspace_dimension:,}')
print(f'gap      {gap:+.4f} mHa')

## B11 - Scan the PES, verify, report

Binding energy is `E(R) - E(48 A)` (paper Eq. 2), so the 48 A point is required,
not optional.

**Do not run `--all` here.** It is 16 distances, each an aug-cc-pVQZ RHF plus an
SQD run, and `reference --all` is 16 CASCI diagonalizations at ~15 GiB apiece:
days of compute. The cell below does the two points `verify` actually needs, and
that is already many hours - but on qBraid the results land in persistent
storage, so you can stop, come back, and widen the grid a point at a time.

In [ ]:
for R in (3.638, 48.000):
    print()
    print('=' * 60)
    print(f'R = {R} A')
    print('=' * 60)
    !{sys.executable} run.py sqd       --distance {R} --rung extrapolation-low
    !{sys.executable} run.py reference --distance {R}

In [ ]:
!{sys.executable} run.py verify
!{sys.executable} run.py report

`report.md` and `report.html` are written into `~/qubit_run/studies/methane-dimer-36q/`.
Open them from the file browser on the left.

To widen the grid later, add distances one at a time - each is resumable and the
cache makes repeats free:

```bash
python run.py sqd       --distance 4.000 --rung extrapolation-low
python run.py reference --distance 4.000
```